In [1]:
!pip install torch transformers datasets seqeval gradio numpy pandas matplotlib scikit-learn torchcrf gensim -q

In [2]:
%%writefile config.py
DATASET_NAME = "conll2003"
LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

EMBEDDING_DIM = 100
CHAR_EMBEDDING_DIM = 50
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.5
MAX_VOCAB_SIZE = 50000
MAX_SEQ_LEN = 128
CHAR_MAX_LEN = 20

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15
CLIP_GRAD = 5.0

TRANSFORMER_MODEL = "distilbert-base-uncased"
TRANSFORMER_LR = 2e-5
TRANSFORMER_EPOCHS = 3
TRANSFORMER_BATCH_SIZE = 16

DEVICE = "cuda"

Overwriting config.py


# Data Loader

In [14]:

%%writefile data_loader.py

import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from collections import Counter

from config import (
    LABEL_LIST, LABEL2ID, ID2LABEL, MAX_VOCAB_SIZE,
    MAX_SEQ_LEN, CHAR_MAX_LEN, BATCH_SIZE
)


def load_conll2003():
    print("Loading CoNLL-2003 dataset...")

    dataset = load_dataset(
        "lhoestq/conll2003",
        trust_remote_code=False
    )

    return dataset


def build_vocab(dataset):
    counter = Counter()
    for example in dataset["train"]:
        for token in example["tokens"]:
            counter[token.lower()] += 1

    vocab = {"<PAD>": 0, "<UNK>": 1}

    for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2):
        vocab[word] = len(vocab)

    return vocab


def build_char_vocab(dataset):
    chars = set()
    for example in dataset["train"]:
        for token in example["tokens"]:
            for ch in token:
                chars.add(ch)

    char_vocab = {"<PAD>": 0, "<UNK>": 1}

    for ch in sorted(chars):
        char_vocab[ch] = len(char_vocab)

    return char_vocab


def convert_ner_tags(ner_tags):
    return ner_tags


class NERDataset(Dataset):

    def __init__(
        self,
        data,
        vocab,
        char_vocab,
        max_seq_len=MAX_SEQ_LEN,
        char_max_len=CHAR_MAX_LEN
    ):

        self.data = data
        self.vocab = vocab
        self.char_vocab = char_vocab
        self.max_seq_len = max_seq_len
        self.char_max_len = char_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        example = self.data[idx]

        tokens = example["tokens"][:self.max_seq_len]

        labels = convert_ner_tags(
            example["ner_tags"][:self.max_seq_len]
        )

        word_ids = [
            self.vocab.get(
                tok.lower(),
                self.vocab["<UNK>"]
            )
            for tok in tokens
        ]

        char_ids = []

        for tok in tokens:

            tok_chars = [
                self.char_vocab.get(
                    ch,
                    self.char_vocab["<UNK>"]
                )
                for ch in tok[:self.char_max_len]
            ]

            tok_chars += [
                0
            ] * (self.char_max_len - len(tok_chars))

            char_ids.append(tok_chars)

        seq_len = len(tokens)

        return {
            "word_ids": torch.tensor(word_ids, dtype=torch.long),
            "char_ids": torch.tensor(char_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "seq_len": seq_len
        }


def collate_fn(batch):

    max_len = max(
        1,
        max(item["seq_len"] for item in batch)
    )

    char_max_len = batch[0]["char_ids"].shape[1]

    batch_size = len(batch)

    word_ids_padded = torch.zeros(
        batch_size, max_len, dtype=torch.long
    )

    char_ids_padded = torch.zeros(
        batch_size, max_len, char_max_len, dtype=torch.long
    )

    labels_padded = torch.full(
        (batch_size, max_len), -1, dtype=torch.long
    )

    seq_lens = []

    masks = torch.zeros(
        batch_size, max_len, dtype=torch.bool
    )

    for i, item in enumerate(batch):

        length = item["seq_len"]

        word_ids_padded[i, :length] = item["word_ids"]

        char_ids_padded[i, :length] = item["char_ids"]

        labels_padded[i, :length] = item["labels"]

        seq_lens.append(length)

        masks[i, :length] = True

    return {
        "word_ids": word_ids_padded,
        "char_ids": char_ids_padded,
        "labels": labels_padded,
        "seq_lens": torch.tensor(seq_lens),
        "masks": masks
    }


def get_dataloaders(dataset, vocab, char_vocab):

    train_dataset = NERDataset(
        dataset["train"], vocab, char_vocab
    )

    val_dataset = NERDataset(
        dataset["validation"], vocab, char_vocab
    )

    test_dataset = NERDataset(
        dataset["test"], vocab, char_vocab
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    return train_loader, val_loader, test_loader


Overwriting data_loader.py


# Embeddings

In [4]:
%%writefile embeddings.py
import numpy as np
import torch
import torch.nn as nn
from config import EMBEDDING_DIM


def load_glove_embeddings(glove_path, vocab, embedding_dim=EMBEDDING_DIM):
    embedding_matrix = np.zeros((len(vocab), embedding_dim))
    found = 0

    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            if word in vocab:
                vector = np.array(parts[1:], dtype=np.float32)
                if len(vector) == embedding_dim:
                    embedding_matrix[vocab[word]] = vector
                    found += 1

    print(f"GloVe: found {found}/{len(vocab)} words")
    return torch.tensor(embedding_matrix, dtype=torch.float32)


def build_embedding_layer(vocab, embedding_dim=EMBEDDING_DIM, pretrained_weights=None, freeze=False):
    embedding = nn.Embedding(len(vocab), embedding_dim, padding_idx=0)

    if pretrained_weights is not None:
        embedding.weight.data.copy_(pretrained_weights)
        if freeze:
            embedding.weight.requires_grad = False

    return embedding

Overwriting embeddings.py


# Models

In [5]:

%%writefile models.py

import torch
import torch.nn as nn

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)

from torchcrf import CRF

from config import (
    EMBEDDING_DIM,
    CHAR_EMBEDDING_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT,
    NUM_LABELS
)


class CharCNNEmbedding(nn.Module):

    def __init__(
        self,
        char_vocab_size,
        char_embed_dim=CHAR_EMBEDDING_DIM,
        num_filters=50,
        kernel_size=3
    ):
        super().__init__()

        self.char_embedding = nn.Embedding(
            char_vocab_size,
            char_embed_dim,
            padding_idx=0
        )

        self.conv = nn.Conv1d(
            char_embed_dim,
            num_filters,
            kernel_size,
            padding=kernel_size // 2
        )

        self.relu = nn.ReLU()

        self.output_dim = num_filters

    def forward(self, char_ids):

        batch_size, seq_len, char_len = char_ids.shape

        char_ids = char_ids.reshape(
            batch_size * seq_len,
            char_len
        )

        char_embeds = self.char_embedding(char_ids)

        char_embeds = char_embeds.permute(0, 2, 1)

        conv_out = self.relu(self.conv(char_embeds))

        pooled = conv_out.max(dim=2).values

        return pooled.reshape(
            batch_size,
            seq_len,
            self.output_dim
        )


class LSTMModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        char_vocab_size,
        num_labels=NUM_LABELS,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pretrained_embeddings=None
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        if pretrained_embeddings is not None:
            self.word_embedding.weight.data.copy_(
                pretrained_embeddings
            )

        self.char_cnn = CharCNNEmbedding(char_vocab_size)

        input_dim = embedding_dim + self.char_cnn.output_dim

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, num_labels)

    def forward(self, word_ids, char_ids, seq_lens=None):

        word_embeds = self.dropout(
            self.word_embedding(word_ids)
        )

        char_embeds = self.char_cnn(char_ids)

        embeds = torch.cat(
            [word_embeds, char_embeds],
            dim=-1
        )

        if seq_lens is not None:

            packed = pack_padded_sequence(
                embeds,
                seq_lens.cpu(),
                batch_first=True,
                enforce_sorted=False
            )

            packed_out, _ = self.lstm(packed)

            lstm_out, _ = pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=word_ids.size(1)
            )

        else:
            lstm_out, _ = self.lstm(embeds)

        lstm_out = self.dropout(lstm_out)

        logits = self.fc(lstm_out)

        return logits


class BiLSTMModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        char_vocab_size,
        num_labels=NUM_LABELS,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pretrained_embeddings=None
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        if pretrained_embeddings is not None:
            self.word_embedding.weight.data.copy_(
                pretrained_embeddings
            )

        self.char_cnn = CharCNNEmbedding(char_vocab_size)

        input_dim = embedding_dim + self.char_cnn.output_dim

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_labels
        )

    def forward(self, word_ids, char_ids, seq_lens=None):

        word_embeds = self.dropout(
            self.word_embedding(word_ids)
        )

        char_embeds = self.char_cnn(char_ids)

        embeds = torch.cat(
            [word_embeds, char_embeds],
            dim=-1
        )

        if seq_lens is not None:

            packed = pack_padded_sequence(
                embeds,
                seq_lens.cpu(),
                batch_first=True,
                enforce_sorted=False
            )

            packed_out, _ = self.lstm(packed)

            lstm_out, _ = pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=word_ids.size(1)
            )

        else:
            lstm_out, _ = self.lstm(embeds)

        lstm_out = self.dropout(lstm_out)

        logits = self.fc(lstm_out)

        return logits


class BiLSTMCRFModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        char_vocab_size,
        num_labels=NUM_LABELS,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        pretrained_embeddings=None
    ):
        super().__init__()

        self.word_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        if pretrained_embeddings is not None:
            self.word_embedding.weight.data.copy_(
                pretrained_embeddings
            )

        self.char_cnn = CharCNNEmbedding(char_vocab_size)

        input_dim = embedding_dim + self.char_cnn.output_dim

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_labels
        )

        self.crf = CRF(
            num_labels,
            batch_first=True
        )

    def forward(self, word_ids, char_ids, masks=None):

        word_embeds = self.dropout(
            self.word_embedding(word_ids)
        )

        char_embeds = self.char_cnn(char_ids)

        embeds = torch.cat(
            [word_embeds, char_embeds],
            dim=-1
        )

        seq_lens = masks.sum(dim=1).cpu()

        packed = pack_padded_sequence(
            embeds,
            seq_lens,
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)

        lstm_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=word_ids.size(1)
        )

        lstm_out = self.dropout(lstm_out)

        emissions = self.fc(lstm_out)

        return emissions

    def loss(self, word_ids, char_ids, labels, masks):

        emissions = self.forward(
            word_ids,
            char_ids,
            masks
        )
        labels_clamped = labels.masked_fill(
            ~masks,
            0
        )

        return -self.crf(
            emissions,
            labels_clamped,
            mask=masks,
            reduction="mean"
        )

    def decode(self, word_ids, char_ids, masks):

        emissions = self.forward(
            word_ids,
            char_ids,
            masks
        )

        return self.crf.decode(
            emissions,
            mask=masks
        )


Overwriting models.py


# Trainer

In [6]:

%%writefile trainer.py

import torch
import torch.nn as nn
import torch.optim as optim

from seqeval.metrics import classification_report, f1_score

from config import (
    LEARNING_RATE,
    NUM_EPOCHS,
    CLIP_GRAD,
    ID2LABEL
)


def get_device():

    return torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


def ids_to_labels(id_seq, length):

    return [
        ID2LABEL[i]
        for i in id_seq[:length]
    ]


def train_epoch(model, loader, optimizer, device, is_crf=False):

    model.train()

    total_loss = 0

    criterion = nn.CrossEntropyLoss(
        ignore_index=-1
    )

    for batch in loader:

        word_ids = batch["word_ids"].to(device)
        char_ids = batch["char_ids"].to(device)
        labels = batch["labels"].to(device)
        masks = batch["masks"].to(device)
        seq_lens = batch["seq_lens"]

        optimizer.zero_grad()

        if is_crf:

            loss = model.loss(
                word_ids,
                char_ids,
                labels,
                masks
            )

        else:

            logits = model(
                word_ids,
                char_ids,
                seq_lens
            )

            batch_size, seq_len, num_labels = logits.shape

            loss = criterion(
                logits.reshape(-1, num_labels),
                labels.reshape(-1)
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            CLIP_GRAD
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device, is_crf=False):

    model.eval()

    all_preds = []
    all_labels = []

    criterion = nn.CrossEntropyLoss(
        ignore_index=-1
    )

    total_loss = 0

    with torch.no_grad():

        for batch in loader:

            word_ids = batch["word_ids"].to(device)
            char_ids = batch["char_ids"].to(device)
            labels = batch["labels"].to(device)
            masks = batch["masks"].to(device)
            seq_lens = batch["seq_lens"]

            if is_crf:

                loss = model.loss(
                    word_ids,
                    char_ids,
                    labels,
                    masks
                )

                total_loss += loss.item()

                decoded = model.decode(
                    word_ids,
                    char_ids,
                    masks
                )

                for i, pred_seq in enumerate(decoded):

                    length = seq_lens[i].item()

                    all_preds.append(
                        ids_to_labels(pred_seq, length)
                    )

                    all_labels.append(
                        ids_to_labels(
                            labels[i].cpu().tolist(),
                            length
                        )
                    )

            else:

                logits = model(
                    word_ids,
                    char_ids,
                    seq_lens
                )

                batch_size, seq_len, num_labels = logits.shape

                loss = criterion(
                    logits.reshape(-1, num_labels),
                    labels.reshape(-1)
                )

                total_loss += loss.item()

                preds = logits.argmax(dim=-1)

                for i in range(batch_size):

                    length = seq_lens[i].item()

                    all_preds.append(
                        ids_to_labels(
                            preds[i].cpu().tolist(),
                            length
                        )
                    )

                    all_labels.append(
                        ids_to_labels(
                            labels[i].cpu().tolist(),
                            length
                        )
                    )

    f1 = f1_score(
        all_labels,
        all_preds
    )

    report = classification_report(
        all_labels,
        all_preds,
        zero_division=0
    )

    return (
        total_loss / len(loader),
        f1,
        report,
        all_preds,
        all_labels
    )


def train_model(
    model,
    train_loader,
    val_loader,
    model_name="model",
    is_crf=False
):

    device = get_device()

    model = model.to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        patience=2,
        factor=0.5
    )

    best_f1 = -1

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_f1": []
    }

    for epoch in range(NUM_EPOCHS):

        train_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            is_crf
        )

        val_loss, val_f1, report, _, _ = evaluate(
            model,
            val_loader,
            device,
            is_crf
        )

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)

        print(
            f"[{model_name}] "
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_f1:

            best_f1 = val_f1

            torch.save(
                model.state_dict(),
                f"{model_name}_best.pt"
            )

            print(
                f"  Saved best model with F1: {best_f1:.4f}"
            )

    model.load_state_dict(
        torch.load(
            f"{model_name}_best.pt",
            map_location=device,
            weights_only=True
        )
    )

    return model, history


Overwriting trainer.py


# Transformer NER

In [29]:
%%writefile transformer_ner.py
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report, f1_score
from config import (
    LABEL_LIST, LABEL2ID, ID2LABEL, NUM_LABELS,
    TRANSFORMER_MODEL, TRANSFORMER_LR,
    TRANSFORMER_EPOCHS, TRANSFORMER_BATCH_SIZE
)


def tokenize_and_align_labels(examples, tokenizer):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding=False
    )

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            prev_word_id = word_id
        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


def compute_metrics_fn(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_label_row = []
        true_pred_row = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_label_row.append(ID2LABEL[l])
                true_pred_row.append(ID2LABEL[p])
        true_labels.append(true_label_row)
        true_preds.append(true_pred_row)

    f1 = f1_score(true_labels, true_preds)
    return {"f1": f1}


def train_transformer(dataset):
    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)

    tokenized_dataset = dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=dataset["train"].column_names
    )

    model = AutoModelForTokenClassification.from_pretrained(
        TRANSFORMER_MODEL,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)

    training_args = TrainingArguments(
        output_dir="transformer_ner_output",
        num_train_epochs=TRANSFORMER_EPOCHS,
        per_device_train_batch_size=TRANSFORMER_BATCH_SIZE,
        per_device_eval_batch_size=TRANSFORMER_BATCH_SIZE,
        learning_rate=TRANSFORMER_LR,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        warmup_steps=100,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
    )

    trainer.train()
    trainer.save_model("transformer_ner_final")
    tokenizer.save_pretrained("transformer_ner_final")

    return model, tokenizer, trainer


def evaluate_transformer(trainer, dataset, tokenizer):
    tokenized_test = dataset["test"].map(
        lambda x: tokenize_and_align_labels(x, tokenizer),
        batched=True,
        remove_columns=dataset["test"].column_names
    )

    predictions, labels, _ = trainer.predict(tokenized_test)
    preds = np.argmax(predictions, axis=-1)

    true_labels = []
    true_preds = []

    for pred_seq, label_seq in zip(preds, labels):
        true_label_row = []
        true_pred_row = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                true_label_row.append(ID2LABEL[l])
                true_pred_row.append(ID2LABEL[p])
        true_labels.append(true_label_row)
        true_preds.append(true_pred_row)

    report = classification_report(true_labels, true_preds, zero_division=0)
    return report, true_preds, true_labels

Overwriting transformer_ner.py


# Evaluate

In [8]:
%%writefile evaluate.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from seqeval.metrics import classification_report
from config import ID2LABEL


def print_report(report_str, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"  {model_name} - Classification Report")
    print(f"{'='*60}")
    print(report_str)


def plot_training_history(histories, save_path="training_history.png"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0f1117")
    colors = ["#6366f1", "#34d399", "#fbbf24", "#f87171"]

    for ax in axes:
        ax.set_facecolor("#1e2235")
        ax.tick_params(colors="white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#2d3154")

    for i, (name, hist) in enumerate(histories.items()):
        color = colors[i % len(colors)]
        axes[0].plot(hist["train_loss"], label=f"{name} train", color=color, linewidth=2)
        axes[0].plot(hist["val_loss"], label=f"{name} val", color=color, linewidth=2, linestyle="--")
        axes[1].plot(hist["val_f1"], label=name, color=color, linewidth=2)

    axes[0].set_title("Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend(facecolor="#1e2235", labelcolor="white")
    axes[1].set_title("Validation F1 Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1")
    axes[1].legend(facecolor="#1e2235", labelcolor="white")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"Training history saved to {save_path}")


def compare_models(results_dict, save_path="model_comparison.png"):
    entity_types = ["PER", "ORG", "LOC", "MISC"]
    metrics = ["precision", "recall", "f1-score"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.patch.set_facecolor("#0f1117")
    colors = ["#6366f1", "#34d399", "#fbbf24", "#f87171"]
    model_names = list(results_dict.keys())
    x = range(len(entity_types))
    bar_width = 0.2

    for ax in axes:
        ax.set_facecolor("#1e2235")
        ax.tick_params(colors="white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#2d3154")

    for metric_idx, metric in enumerate(metrics):
        ax = axes[metric_idx]
        for model_idx, model_name in enumerate(model_names):
            report_dict = results_dict[model_name]
            values = [report_dict.get(entity, {}).get(metric, 0) for entity in entity_types]
            offset = (model_idx - len(model_names) / 2) * bar_width
            ax.bar(
                [xi + offset for xi in x],
                values,
                width=bar_width,
                label=model_name,
                color=colors[model_idx % len(colors)],
                alpha=0.85
            )

        ax.set_title(metric.capitalize())
        ax.set_xticks(list(x))
        ax.set_xticklabels(entity_types, color="white")
        ax.set_ylim(0, 1)
        ax.legend(facecolor="#1e2235", labelcolor="white")

    plt.suptitle("Model Comparison by Entity Type", color="white", fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"Comparison chart saved to {save_path}")


def parse_report_to_dict(report_str):
    lines = report_str.strip().split("\n")
    result = {}
    for line in lines:
        parts = line.split()
        if len(parts) >= 5 and parts[0] not in ["precision", "micro", "macro", "weighted"]:
            entity = parts[0]
            try:
                result[entity] = {
                    "precision": float(parts[1]),
                    "recall": float(parts[2]),
                    "f1-score": float(parts[3]),
                    "support": int(parts[4])
                }
            except ValueError:
                continue
    return result

Overwriting evaluate.py


# Train All Models

In [10]:
!pip install pytorch-crf -q

In [11]:
from torchcrf import CRF
print("torchcrf imported successfully")

torchcrf imported successfully


In [15]:
import importlib
import data_loader

importlib.reload(data_loader)

from data_loader import (
    load_conll2003,
    build_vocab,
    build_char_vocab,
    get_dataloaders
)

In [16]:
from datasets import load_dataset

print("Loading CoNLL-2003 dataset...")

dataset = load_dataset(
    "lhoestq/conll2003",
    trust_remote_code=False
)

print(dataset)

Loading CoNLL-2003 dataset...
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [17]:
from data_loader import (
    load_conll2003,
    build_vocab,
    build_char_vocab,
    get_dataloaders
)
print("Loading dataset...")
dataset = load_conll2003()

print("Building vocabularies...")
vocab = build_vocab(dataset)
char_vocab = build_char_vocab(dataset)

print("Vocab size:", len(vocab))
print("Character vocab size:", len(char_vocab))

print("Building dataloaders...")
train_loader, val_loader, test_loader = get_dataloaders(
    dataset, vocab, char_vocab
)


batch = next(iter(train_loader))

print("\nBatch shapes:")
print("Word IDs:", batch["word_ids"].shape)
print("Character IDs:", batch["char_ids"].shape)
print("Labels:", batch["labels"].shape)
print("Masks:", batch["masks"].shape)

print("\nData loader is working!")

Loading dataset...
Loading CoNLL-2003 dataset...
Building vocabularies...
Vocab size: 21011
Character vocab size: 86
Building dataloaders...

Batch shapes:
Word IDs: torch.Size([32, 38])
Character IDs: torch.Size([32, 38, 20])
Labels: torch.Size([32, 38])
Masks: torch.Size([32, 38])

Data loader is working!


In [18]:
batch = next(iter(train_loader))

print("Word IDs:", batch["word_ids"].min().item(),
      batch["word_ids"].max().item())

print("Labels:", batch["labels"].min().item(),
      batch["labels"].max().item())

print("Sequence lengths:", batch["seq_lens"][:10])
print("Number of real tokens:", batch["masks"].sum().item())

Word IDs: 0 20976
Labels: -1 8
Sequence lengths: tensor([ 1, 26, 11,  3,  4, 31, 20,  7,  4, 22])
Number of real tokens: 471


# Train LSTM

In [22]:
import torch
import os
from data_loader import load_conll2003, build_vocab, build_char_vocab, get_dataloaders
from models import LSTMModel, BiLSTMModel, BiLSTMCRFModel
from trainer import train_model, evaluate, get_device
from transformer_ner import train_transformer, evaluate_transformer
from evaluate import print_report, plot_training_history, compare_models, parse_report_to_dict
from config import NUM_LABELS

device = get_device()
print("Using device:", device)

histories = {}
test_reports = {}

print("\n--- Training LSTM ---")
lstm_model = LSTMModel(vocab_size=len(vocab), char_vocab_size=len(char_vocab), num_labels=NUM_LABELS)
lstm_model, lstm_history = train_model(lstm_model, train_loader, val_loader, "lstm")
histories["LSTM"] = lstm_history
_, _, lstm_report, _, _ = evaluate(lstm_model.to(device), test_loader, device, is_crf=False)
print_report(lstm_report, "LSTM")
test_reports["LSTM"] = parse_report_to_dict(lstm_report)

Using device: cuda

--- Training LSTM ---
[lstm] Epoch 1/15 | Train Loss: 0.4998 | Val Loss: 0.3118 | Val F1: 0.3514
  Saved best model with F1: 0.3514
[lstm] Epoch 2/15 | Train Loss: 0.2561 | Val Loss: 0.2285 | Val F1: 0.5101
  Saved best model with F1: 0.5101
[lstm] Epoch 3/15 | Train Loss: 0.2080 | Val Loss: 0.1863 | Val F1: 0.6240
  Saved best model with F1: 0.6240
[lstm] Epoch 4/15 | Train Loss: 0.1740 | Val Loss: 0.1607 | Val F1: 0.6944
  Saved best model with F1: 0.6944
[lstm] Epoch 5/15 | Train Loss: 0.1514 | Val Loss: 0.1408 | Val F1: 0.7158
  Saved best model with F1: 0.7158
[lstm] Epoch 6/15 | Train Loss: 0.1339 | Val Loss: 0.1318 | Val F1: 0.7351
  Saved best model with F1: 0.7351
[lstm] Epoch 7/15 | Train Loss: 0.1208 | Val Loss: 0.1237 | Val F1: 0.7481
  Saved best model with F1: 0.7481
[lstm] Epoch 8/15 | Train Loss: 0.1107 | Val Loss: 0.1200 | Val F1: 0.7524
  Saved best model with F1: 0.7524
[lstm] Epoch 9/15 | Train Loss: 0.1007 | Val Loss: 0.1163 | Val F1: 0.7635
  S

# Train BiLSTM

In [27]:
print("\n--- Training BiLSTM ---")
bilstm_model = BiLSTMModel(vocab_size=len(vocab), char_vocab_size=len(char_vocab), num_labels=NUM_LABELS)
bilstm_model, bilstm_history = train_model(bilstm_model, train_loader, val_loader, "bilstm")
histories["BiLSTM"] = bilstm_history
_, _, bilstm_report, _, _ = evaluate(bilstm_model.to(device), test_loader, device, is_crf=False)
print_report(bilstm_report, "BiLSTM")
test_reports["BiLSTM"] = parse_report_to_dict(bilstm_report)


--- Training BiLSTM ---
[bilstm] Epoch 1/15 | Train Loss: 0.4127 | Val Loss: 0.1842 | Val F1: 0.6416
  Saved best model with F1: 0.6416
[bilstm] Epoch 2/15 | Train Loss: 0.1679 | Val Loss: 0.1316 | Val F1: 0.7438
  Saved best model with F1: 0.7438
[bilstm] Epoch 3/15 | Train Loss: 0.1239 | Val Loss: 0.1003 | Val F1: 0.8079
  Saved best model with F1: 0.8079
[bilstm] Epoch 4/15 | Train Loss: 0.0975 | Val Loss: 0.0916 | Val F1: 0.8229
  Saved best model with F1: 0.8229
[bilstm] Epoch 5/15 | Train Loss: 0.0802 | Val Loss: 0.0896 | Val F1: 0.8336
  Saved best model with F1: 0.8336
[bilstm] Epoch 6/15 | Train Loss: 0.0667 | Val Loss: 0.0907 | Val F1: 0.8396
  Saved best model with F1: 0.8396
[bilstm] Epoch 7/15 | Train Loss: 0.0564 | Val Loss: 0.0868 | Val F1: 0.8532
  Saved best model with F1: 0.8532
[bilstm] Epoch 8/15 | Train Loss: 0.0493 | Val Loss: 0.0835 | Val F1: 0.8574
  Saved best model with F1: 0.8574
[bilstm] Epoch 9/15 | Train Loss: 0.0427 | Val Loss: 0.0831 | Val F1: 0.8670
  

# Train BiLSTM + CRF

In [24]:
print("\n--- Training BiLSTM + CRF ---")
bilstm_crf_model = BiLSTMCRFModel(vocab_size=len(vocab), char_vocab_size=len(char_vocab), num_labels=NUM_LABELS)
bilstm_crf_model, crf_history = train_model(bilstm_crf_model, train_loader, val_loader, "bilstm_crf", is_crf=True)
histories["BiLSTM+CRF"] = crf_history
_, _, crf_report, _, _ = evaluate(bilstm_crf_model.to(device), test_loader, device, is_crf=True)
print_report(crf_report, "BiLSTM+CRF")
test_reports["BiLSTM+CRF"] = parse_report_to_dict(crf_report)


--- Training BiLSTM + CRF ---
[bilstm_crf] Epoch 1/15 | Train Loss: 5.1092 | Val Loss: 2.4491 | Val F1: 0.7082
  Saved best model with F1: 0.7082
[bilstm_crf] Epoch 2/15 | Train Loss: 1.9809 | Val Loss: 1.6518 | Val F1: 0.7923
  Saved best model with F1: 0.7923
[bilstm_crf] Epoch 3/15 | Train Loss: 1.4334 | Val Loss: 1.3884 | Val F1: 0.8285
  Saved best model with F1: 0.8285
[bilstm_crf] Epoch 4/15 | Train Loss: 1.1199 | Val Loss: 1.2387 | Val F1: 0.8437
  Saved best model with F1: 0.8437
[bilstm_crf] Epoch 5/15 | Train Loss: 0.9120 | Val Loss: 1.2710 | Val F1: 0.8427
[bilstm_crf] Epoch 6/15 | Train Loss: 0.7649 | Val Loss: 1.1384 | Val F1: 0.8623
  Saved best model with F1: 0.8623
[bilstm_crf] Epoch 7/15 | Train Loss: 0.6422 | Val Loss: 1.1231 | Val F1: 0.8718
  Saved best model with F1: 0.8718
[bilstm_crf] Epoch 8/15 | Train Loss: 0.5602 | Val Loss: 1.1378 | Val F1: 0.8673
[bilstm_crf] Epoch 9/15 | Train Loss: 0.4982 | Val Loss: 1.1663 | Val F1: 0.8626
[bilstm_crf] Epoch 10/15 | Tra

# Train DistilBERT

In [31]:
import importlib
import transformer_ner

importlib.reload(transformer_ner)

from transformer_ner import train_transformer, evaluate_transformer

In [32]:
print("\n--- Training DistilBERT ---")
transformer_model, tokenizer, trainer = train_transformer(dataset)
transformer_report, _, _ = evaluate_transformer(trainer, dataset, tokenizer)
print_report(transformer_report, "DistilBERT")
test_reports["DistilBERT"] = parse_report_to_dict(transformer_report)


--- Training DistilBERT ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1
1,0.060790,0.055872,0.915475
2,0.028079,0.049406,0.932164
3,0.024127,0.050056,0.935300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


  DistilBERT - Classification Report
              precision    recall  f1-score   support

         LOC       0.89      0.93      0.91      1667
        MISC       0.75      0.79      0.77       702
         ORG       0.85      0.86      0.86      1661
         PER       0.97      0.96      0.96      1616

   micro avg       0.88      0.90      0.89      5646
   macro avg       0.87      0.89      0.88      5646
weighted avg       0.89      0.90      0.89      5646



In [36]:
from google.colab import drive
import os

MODEL_DIR = "/content/drive/MyDrive/NER_Project/models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [37]:
torch.save(
    lstm_model.state_dict(),
    f"{MODEL_DIR}/lstm.pth"
)

torch.save(
    bilstm_model.state_dict(),
    f"{MODEL_DIR}/bilstm.pth"
)

torch.save(
    bilstm_crf_model.state_dict(),
    f"{MODEL_DIR}/bilstm_crf.pth"
)

In [38]:
torch.save(
    lstm_model.state_dict(),
    f"{MODEL_DIR}/lstm.pth"
)

torch.save(
    bilstm_model.state_dict(),
    f"{MODEL_DIR}/bilstm.pth"
)

torch.save(
    bilstm_crf_model.state_dict(),
    f"{MODEL_DIR}/bilstm_crf.pth"
)

In [39]:
transformer_model.save_pretrained(
    f"{MODEL_DIR}/distilbert"
)

tokenizer.save_pretrained(
    f"{MODEL_DIR}/distilbert"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/NER_Project/models/distilbert/tokenizer_config.json',
 '/content/drive/MyDrive/NER_Project/models/distilbert/tokenizer.json')

In [40]:
torch.save({
    "vocab": vocab,
    "char_vocab": char_vocab
}, f"{MODEL_DIR}/vocab_data.pt")

In [41]:
from google.colab import files

files.download(f"{MODEL_DIR}/lstm.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
import shutil

shutil.make_archive(
    "/content/NER_Models",
    "zip",
    MODEL_DIR
)

files.download("/content/NER_Models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [48]:
import os
import html

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

from models import BiLSTMCRFModel
from config import ID2LABEL, NUM_LABELS


BASE_DIR = "/content"

MODEL_DIR = "/content/drive/MyDrive/NER_Project/models"

TRANSFORMER_DIR = os.path.join(MODEL_DIR, "distilbert")
BILSTM_CRF_PATH = os.path.join(MODEL_DIR, "bilstm_crf.pth")
VOCAB_PATH = os.path.join(MODEL_DIR, "vocab_data.pt")


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 128
MAX_WORD_CHARS = 20

ENTITY_COLORS = {
    "PER": {"bg": "#dbeafe", "border": "#3b82f6", "text": "#1e3a5f"},
    "ORG": {"bg": "#dcfce7", "border": "#22c55e", "text": "#14532d"},
    "LOC": {"bg": "#fef9c3", "border": "#eab308", "text": "#713f12"},
    "MISC": {"bg": "#fce7f3", "border": "#ec4899", "text": "#831843"},
}

transformer_model = None
transformer_tokenizer = None

bilstm_crf_model = None
vocab = None
char_vocab = None


# ---------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------

def load_transformer():
    """Load DistilBERT once and reuse it for later predictions."""
    global transformer_model, transformer_tokenizer

    if transformer_model is not None and transformer_tokenizer is not None:
        return True

    if not os.path.isdir(TRANSFORMER_DIR):
        return False

    try:
        transformer_tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_DIR)
        transformer_model = AutoModelForTokenClassification.from_pretrained(
            TRANSFORMER_DIR
        )
        transformer_model.to(DEVICE)
        transformer_model.eval()
        return True
    except Exception as exc:
        print(f"Could not load Transformer model: {exc}")
        transformer_model = None
        transformer_tokenizer = None
        return False


def load_bilstm_crf():
    """Load BiLSTM-CRF weights and vocabularies once."""
    global bilstm_crf_model, vocab, char_vocab

    if bilstm_crf_model is not None:
        return True

    if not (os.path.isfile(BILSTM_CRF_PATH) and os.path.isfile(VOCAB_PATH)):
        return False

    try:
        # weights_only=False is needed because this file contains Python
        # dictionaries (vocabularies), not only tensors.
        saved = torch.load(VOCAB_PATH, map_location="cpu", weights_only=False)
        vocab = saved["vocab"]
        char_vocab = saved["char_vocab"]

        bilstm_crf_model = BiLSTMCRFModel(
            vocab_size=len(vocab),
            char_vocab_size=len(char_vocab),
            num_labels=NUM_LABELS,
        )

        state_dict = torch.load(
            BILSTM_CRF_PATH,
            map_location="cpu",
            weights_only=True,
        )
        bilstm_crf_model.load_state_dict(state_dict)
        bilstm_crf_model.to(DEVICE)
        bilstm_crf_model.eval()
        return True

    except Exception as exc:
        print(f"Could not load BiLSTM-CRF model: {exc}")
        bilstm_crf_model = None
        vocab = None
        char_vocab = None
        return False


# ---------------------------------------------------------------------
# Prediction
# ---------------------------------------------------------------------

def predict_transformer(text):
    if transformer_model is None or transformer_tokenizer is None:
        return None

    tokens = text.strip().split()
    if not tokens:
        return []

    encoding = transformer_tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    model_inputs = {key: value.to(DEVICE) for key, value in encoding.items()}

    with torch.no_grad():
        outputs = transformer_model(**model_inputs)

    predictions = outputs.logits[0].argmax(dim=-1).tolist()
    word_ids = encoding.word_ids(batch_index=0)

    token_labels = []
    seen_word_ids = set()

    for token_index, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen_word_ids:
            continue

        seen_word_ids.add(word_id)
        if word_id < len(tokens):
            label_id = predictions[token_index]
            token_labels.append((tokens[word_id], ID2LABEL[label_id]))

    return token_labels


def predict_bilstm_crf(text):
    if bilstm_crf_model is None or vocab is None or char_vocab is None:
        return None

    tokens = text.strip().split()[:MAX_LENGTH]
    if not tokens:
        return []

    unknown_word_id = vocab.get("<UNK>", 0)
    unknown_char_id = char_vocab.get("<UNK>", 0)

    word_ids = [
        vocab.get(token.lower(), unknown_word_id)
        for token in tokens
    ]

    char_ids = []
    for token in tokens:
        token_char_ids = [
            char_vocab.get(char, unknown_char_id)
            for char in token[:MAX_WORD_CHARS]
        ]
        token_char_ids += [0] * (MAX_WORD_CHARS - len(token_char_ids))
        char_ids.append(token_char_ids)

    word_tensor = torch.tensor([word_ids], dtype=torch.long, device=DEVICE)
    char_tensor = torch.tensor([char_ids], dtype=torch.long, device=DEVICE)
    mask = torch.ones((1, len(tokens)), dtype=torch.bool, device=DEVICE)

    with torch.no_grad():
        decoded = bilstm_crf_model.decode(word_tensor, char_tensor, mask)

    predictions = decoded[0]
    return [
        (tokens[index], ID2LABEL[int(predictions[index])])
        for index in range(len(tokens))
    ]


# ---------------------------------------------------------------------
# HTML rendering
# ---------------------------------------------------------------------

def tokens_to_html(token_labels):
    if not token_labels:
        return "<p style='color:#94a3b8;'>No tokens to display.</p>"

    html_parts = []
    index = 0

    while index < len(token_labels):
        token, label = token_labels[index]
        safe_token = html.escape(str(token))

        if label == "O":
            html_parts.append(
                f"<span style='color:#f1f5f9;font-size:16px;'>{safe_token} </span>"
            )
            index += 1
            continue

        if "-" in label:
            prefix, entity_type = label.split("-", 1)
        else:
            prefix, entity_type = "B", label

        # An I-tag without a preceding B-tag is displayed as ordinary text.
        if prefix != "B":
            html_parts.append(
                f"<span style='color:#f1f5f9;font-size:16px;'>{safe_token} </span>"
            )
            index += 1
            continue

        span_tokens = [str(token)]
        next_index = index + 1

        while next_index < len(token_labels):
            next_token, next_label = token_labels[next_index]
            if next_label == f"I-{entity_type}":
                span_tokens.append(str(next_token))
                next_index += 1
            else:
                break

        colors = ENTITY_COLORS.get(
            entity_type,
            {"bg": "#e2e8f0", "border": "#94a3b8", "text": "#1e293b"},
        )

        safe_span = html.escape(" ".join(span_tokens))
        safe_type = html.escape(entity_type)

        html_parts.append(
            f"<span style='"
            f"background:{colors['bg']};"
            f"border:1.5px solid {colors['border']};"
            f"color:{colors['text']};"
            f"border-radius:5px;"
            f"padding:2px 6px;"
            f"margin:0 3px;"
            f"font-size:15px;"
            f"font-weight:500;"
            f"'>"
            f"{safe_span}"
            f"<sup style='font-size:10px;margin-left:3px;font-weight:700;"
            f"color:{colors['border']};'>{safe_type}</sup>"
            f"</span> "
        )

        index = next_index

    return (
        "<div style='line-height:2.4;padding:16px;'>"
        + "".join(html_parts)
        + "</div>"
    )


def extract_entities(token_labels):
    """Convert BIO-tagged tokens into a dictionary grouped by entity type."""
    entities = {}
    index = 0

    while index < len(token_labels):
        token, label = token_labels[index]

        if label.startswith("B-"):
            entity_type = label[2:]
            span = [str(token)]
            next_index = index + 1

            while (
                next_index < len(token_labels)
                and token_labels[next_index][1] == f"I-{entity_type}"
            ):
                span.append(str(token_labels[next_index][0]))
                next_index += 1

            entities.setdefault(entity_type, []).append(" ".join(span))
            index = next_index
        else:
            index += 1

    return entities


def create_summary_html(token_labels):
    entities = extract_entities(token_labels)

    if not entities:
        return (
            "<div style='padding:12px;background:#1e2235;border-radius:8px;"
            "border:1px solid #2d3154;color:#94a3b8;'>No entities found.</div>"
        )

    summary_lines = []

    for entity_type, entity_values in sorted(entities.items()):
        colors = ENTITY_COLORS.get(entity_type, {"border": "#94a3b8"})
        safe_type = html.escape(entity_type)
        safe_values = ", ".join(
            f"<b>{html.escape(value)}</b>" for value in entity_values
        )

        summary_lines.append(
            "<div style='margin:4px 0;'>"
            f"<span style='color:{colors['border']};font-weight:700;'>"
            f"{safe_type}</span>: "
            f"<span style='color:#e2e8f0;'>{safe_values}</span>"
            "</div>"
        )

    return (
        "<div style='padding:12px;background:#1e2235;border-radius:8px;"
        "border:1px solid #2d3154;'>"
        + "".join(summary_lines)
        + "</div>"
    )


# ---------------------------------------------------------------------
# Gradio event function
# ---------------------------------------------------------------------

def run_ner(text, model_choice):
    if not text or not text.strip():
        return (
            "<p style='color:#94a3b8;'>Please enter some text.</p>",
            "",
        )

    if model_choice == "DistilBERT (Transformer)":
        if not load_transformer():
            return (
                "<p style='color:#f87171;'>Transformer model files were not found. "
                "Check the transformer_ner_final folder.</p>",
                "",
            )
        token_labels = predict_transformer(text)

    elif model_choice == "BiLSTM + CRF":
        if not load_bilstm_crf():
            return (
                "<p style='color:#f87171;'>BiLSTM-CRF model files were not found. "
                "Check bilstm_crf_best.pt and vocab_data.pt.</p>",
                "",
            )
        token_labels = predict_bilstm_crf(text)

    else:
        return (
            "<p style='color:#f87171;'>Unknown model selected.</p>",
            "",
        )

    if token_labels is None:
        return "<p style='color:#f87171;'>Prediction failed.</p>", ""

    return tokens_to_html(token_labels), create_summary_html(token_labels)


# ---------------------------------------------------------------------
# Gradio interface
# ---------------------------------------------------------------------

EXAMPLES = [
    "Apple CEO Tim Cook announced a new partnership with Microsoft in Seattle last Monday.",
    "The United Nations held a summit in Geneva where French President Emmanuel Macron spoke.",
    "Amazon acquired MGM Studios for 8.5 billion dollars according to Reuters.",
    "Elon Musk's SpaceX launched a Falcon 9 rocket from Cape Canaveral in Florida.",
]

CSS = """
body {
    background-color: #0f1117;
}
.gradio-container {
    background-color: #0f1117 !important;
    font-family: 'Inter', sans-serif !important;
    max-width: 960px !important;
    margin: 0 auto !important;
}
"""

with gr.Blocks(css=CSS, title="NER System") as demo:
    gr.HTML(
        """
        <div style='text-align:center;padding:32px 0 16px 0;'>
            <h1 style='color:#f1f5f9;font-size:28px;font-weight:700;margin:0;'>
                🏷️ Named Entity Recognition System
            </h1>
            <p style='color:#94a3b8;font-size:14px;margin:8px 0 0 0;'>
                Identify people, organizations, locations, and miscellaneous entities
            </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            gr.HTML(
                """
                <div style='display:flex;gap:12px;margin-bottom:12px;flex-wrap:wrap;'>
                    <span style='background:#dbeafe;border:1.5px solid #3b82f6;border-radius:4px;padding:1px 8px;font-size:12px;color:#1e3a5f;font-weight:600;'>PER</span>
                    <span style='background:#dcfce7;border:1.5px solid #22c55e;border-radius:4px;padding:1px 8px;font-size:12px;color:#14532d;font-weight:600;'>ORG</span>
                    <span style='background:#fef9c3;border:1.5px solid #eab308;border-radius:4px;padding:1px 8px;font-size:12px;color:#713f12;font-weight:600;'>LOC</span>
                    <span style='background:#fce7f3;border:1.5px solid #ec4899;border-radius:4px;padding:1px 8px;font-size:12px;color:#831843;font-weight:600;'>MISC</span>
                </div>
                """
            )

            text_input = gr.Textbox(
                label="Input Text",
                placeholder="Enter text to analyze for named entities...",
                lines=4,
            )

            model_choice = gr.Radio(
                choices=["DistilBERT (Transformer)", "BiLSTM + CRF"],
                value="DistilBERT (Transformer)",
                label="Select Model",
            )

            with gr.Row():
                submit_btn = gr.Button("Analyze →", variant="primary")
                clear_btn = gr.Button("Clear", variant="secondary")

            gr.HTML(
                "<p style='color:#64748b;font-size:12px;margin-top:4px;'>"
                "Try an example:</p>"
            )

            for example in EXAMPLES:
                gr.Button(example[:70] + "...", size="sm").click(
                    fn=lambda example=example: example,
                    outputs=text_input,
                )

        with gr.Column(scale=4):
            gr.HTML(
                "<p style='color:#94a3b8;font-size:13px;margin:0 0 6px 0;'>"
                "Entity Highlights</p>"
            )
            output_html = gr.HTML(
                value="<div style='padding:16px;color:#64748b;'>"
                "Results will appear here...</div>"
            )

            gr.HTML(
                "<p style='color:#94a3b8;font-size:13px;margin:12px 0 6px 0;'>"
                "Entity Summary</p>"
            )
            summary_html = gr.HTML(
                value="<div style='padding:12px;background:#1e2235;"
                "border-radius:8px;color:#64748b;'>"
                "Summary will appear here...</div>"
            )

    submit_btn.click(
        fn=run_ner,
        inputs=[text_input, model_choice],
        outputs=[output_html, summary_html],
    )

    clear_btn.click(
        fn=lambda: (
            "",
            "<div style='padding:16px;color:#64748b;'>"
            "Results will appear here...</div>",
            "<div style='padding:12px;background:#1e2235;"
            "border-radius:8px;color:#64748b;'>"
            "Summary will appear here...</div>",
        ),
        outputs=[text_input, output_html, summary_html],
    )


if __name__ == "__main__":
    demo.launch(share=True)


/tmp/ipykernel_16319/171739607.py:398: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="NER System") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fa8ea350becde755f5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
